# Native AHR v5 — final validation and detached launch

Purpose: bind successful engineering checks to the exact final sources, record dependencies and launch the authorized GPU experiment. Inputs: completed verified curriculum and current full training/arena recovery receipts. Outputs: immutable delivery validation/protocol, environment record and detached launch receipt. Expected runtime: 2–5 minutes; the separate main run targets 18 hours and has a 24-hour hard limit.

This notebook does not silently start generation or upstream training. All imports use `ahr-export` cells. Launch: fresh `ahr-go` kernel with `AHR_NATIVE_V5_LAUNCH=1`, Run All. Monitor notebook 06. Session `ahr-go-native-v5`; attach `tmux attach -t ahr-go-native-v5`, detach **Ctrl-b, then d**.

Checkpoint locations: `checkpoints/ahr-native-v5-main/`, `checkpoints/pipeline-ahr-native-v5/` and separate arena runs. Exact resume uses notebook 05 with `AHR_NATIVE_V5_RUN=1`, unchanged sources/configuration and the original restored budget. Portable restore requires matching Python/venv dependencies and pinned GNU binary; identities are recorded in `reports/ahr_native_v5/environment.json`. All files stay on this VM per the user's instruction.

In [ ]:
from pathlib import Path
import os, nbformat
PROJECT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks/00_instance_setup_and_recovery.ipynb').is_file())
book=nbformat.read(PROJECT/'experiments/ahr_native_v5/00_shared.ipynb',4)
for c in book.cells:
    if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[]):exec(compile(c.source,'v5 shared','exec'),globals())
v5_load_v4(PROJECT,globals())
for name in V5_NOTEBOOKS[1:]:v5_import_exports(PROJECT/V5_ROOT/name,globals())


## Current engineering evidence

In [ ]:
import platform, subprocess
from IPython.display import display, Markdown
sources=v5_source_identity(PROJECT)
curriculum=v5_curriculum(PROJECT)
expected=V5CurriculumConfig()
expected_identity={'schema':V5_CURRICULUM_SCHEMA,'config':asdict(expected),'definition_sha256':v5_code_sha(PROJECT/V5_CURRICULUM_NOTEBOOK),'upstream':v4_source_identity(PROJECT)}
if curriculum['provenance'].get('identity')!=expected_identity:raise RuntimeError('Curriculum source/configuration identity differs')
for split,quota in [('train',expected.train_per_goal),('development',expected.development_per_goal)]:
    for goal in V5_GOALS:
        if sum(row['tactical_goal']==goal for row in curriculum[split])!=quota:raise RuntimeError('Full balanced independent curriculum required: '+split+'/'+goal)
if any(row.get('policy_weight')!=0 or row['verification'].get('external_engine_labels') is not False for split in ['train','development'] for row in curriculum[split]):raise RuntimeError('Unsupported curriculum policy teacher')
model=v5_validate_model_search(PROJECT,device='cuda:1')
if model.get('sources')!=sources:raise RuntimeError('Model validation sources differ')
v5_atomic_json(v5_report_path(PROJECT,'model_search_validation.json'),model)
controller=v5_validate_controller_recovery(PROJECT)
engineering=v5_validate_gates(PROJECT,model=None,device='cuda:1')
training=v5_read_json(v5_report_path(PROJECT,'training_recovery.json'))
if training.get('status')!='PASS' or training.get('exact_resume') is not True:raise RuntimeError('Actual optimizer recovery proof required')
if training.get('sources',training.get('source_identity'))!=sources:raise RuntimeError('Training recovery sources differ')
generation=v5_read_json(v5_report_path(PROJECT,'curriculum_recovery.json'))
if generation.get('status')!='PASS' or generation.get('sources')!=sources or not generation.get('checks') or not all(generation['checks'].values()):raise RuntimeError('Current complete curriculum recovery proof required')
extensions=v5_read_json(v5_report_path(PROJECT,'learning_recovery_extensions.json'))
if extensions.get('status')!='PASS' or extensions.get('sources')!=sources or not extensions.get('checks') or not all(extensions['checks'].values()):raise RuntimeError('Current native and historical actor recovery proof required')
gate_regression=v5_read_json(v5_report_path(PROJECT,'first_gate_regression.json'))
if gate_regression.get('status')!='PASS' or gate_regression.get('sources')!=sources or not gate_regression.get('checks') or not all(gate_regression['checks'].values()):raise RuntimeError('Current actual full-development learning-gate runtime proof required')
arena=v5_read_json(v5_report_path(PROJECT,'arena_validation.json'))
if arena.get('sources')!=sources or not arena.get('exact_resume'):raise RuntimeError('Current arena recovery proof required')
checks={'model_search':model['status'],'curriculum':'PASS','training_recovery':training['status'],'arena_recovery':arena['status'],'controller_recovery':controller['status'],'engineering_gates':engineering['status'],'curriculum_recovery':generation['status'],'native_actor_recovery':extensions['status'],'learning_gate_runtime':gate_regression['status']}
if any(v!='PASS' for v in checks.values()):raise RuntimeError('A launch check failed')
receipt={'schema':V5_SCHEMA,'status':'PASS','checks':checks,'sources':sources,
         'curriculum':curriculum['provenance'],'model_search':model,'training_recovery':training,'arena_recovery':arena,
         'controller_recovery':controller,'engineering':engineering,'curriculum_recovery':generation,'native_actor_recovery':extensions,'learning_gate_runtime':gate_regression,'validated_at_utc':v5_now(),
         'learning_competence':'Measured during adaptive warmup; failure retains supervised diagnostic phase; no final strength claim'}
path=v5_report_path(PROJECT,'delivery_validation.json')
if (PROJECT/V5_PROTOCOL).exists():
    protocol=v5_protocol_read(PROJECT)
    if v5_read_json(path)['sources']!=sources:raise RuntimeError('Frozen delivery sources differ')
    v5_atomic_json(v5_report_path(PROJECT,'delivery_recheck.json'),receipt)
else:v5_atomic_json(path,receipt)
environment={'python':platform.python_version(),'torch':torch.__version__,'cuda_runtime':torch.version.cuda,
             'gpu':torch.cuda.get_device_name(1),'gnu':arena['gnu_identity'],'storage':'persistent boot disk /dev/sda1; instance-deletion policy unverified',
             'dependencies':subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True).splitlines()}
v5_atomic_json(v5_report_path(PROJECT,'environment.json'),environment)
display(Markdown('| Launch check | Result |\n|---|---|\n'+'\n'.join('| '+k.replace('_',' ')+' | '+v+' |' for k,v in checks.items())))


## Freeze and launch

In [ ]:
if os.environ.get('AHR_NATIVE_V5_LAUNCH')=='1':
    protocol=v5_protocol_prepare(PROJECT)
    launched=v5_launch_detached(PROJECT)
    print({k:launched[k] for k in ('status','session','target_hours','hard_limit_hours','device')})
else:
    print('Validation complete. Authorized launch: AHR_NATIVE_V5_LAUNCH=1; Run All in this notebook.')
